In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import numpy as np


QUANTIZED_MODEL_PATH = Path("models/quantized/cifar_alexnet_int8.npz")
QUANTIZATION_CONFIG_PATH = Path("models/quantized/quantization_config.json")

INTEGER_RESULTS_DIRECTORY = Path("integer_results")
OUTPUT_DIRECTORY = Path("rtl_data")

REQUANT_SHIFT = 24

CONV_LAYERS = ["conv1", "conv2", "conv3", "conv4", "conv5"]
FC_LAYERS = ["fc6", "fc7", "fc8"]
ALL_COMPUTE_LAYERS = CONV_LAYERS + FC_LAYERS

CLASS_NAMES = [
    "airplane",
    "automobile",
    "bird",
    "cat",
    "deer",
    "dog",
    "frog",
    "horse",
    "ship",
    "truck",
]

NUM_CLASSES = len(CLASS_NAMES)

EXPECTED_SHAPES = {
    "conv1_weight": (8, 3, 5, 5),
    "conv1_bias": (8,),

    "conv2_weight": (16, 8, 3, 3),
    "conv2_bias": (16,),

    "conv3_weight": (32, 16, 3, 3),
    "conv3_bias": (32,),

    "conv4_weight": (32, 32, 3, 3),
    "conv4_bias": (32,),

    "conv5_weight": (16, 32, 3, 3),
    "conv5_bias": (16,),

    "fc6_weight": (64, 256),
    "fc6_bias": (64,),

    "fc7_weight": (32, 64),
    "fc7_bias": (32,),

    "fc8_weight": (NUM_CLASSES, 32),
    "fc8_bias": (NUM_CLASSES,),
}


def int8_to_hex(value: int) -> str:
    if not -128 <= value <= 127:
        raise ValueError(f"INT8 value outside range: {value}")

    return f"{value & 0xFF:02x}"


def int32_to_hex(value: int) -> str:
    if not -(2**31) <= value <= (2**31 - 1):
        raise ValueError(f"INT32 value outside range: {value}")

    return f"{value & 0xFFFFFFFF:08x}"


def uint32_to_hex(value: int) -> str:
    if not 0 <= value <= (2**32 - 1):
        raise ValueError(f"UINT32 value outside range: {value}")

    return f"{value:08x}"


def write_int8_mem(output_path: Path, values: np.ndarray) -> None:
    flattened = np.asarray(values, dtype=np.int8).flatten(order="C")

    with output_path.open("w", encoding="utf-8", newline="\n") as file:
        for value in flattened:
            file.write(int8_to_hex(int(value)) + "\n")


def write_int32_mem(output_path: Path, values: np.ndarray) -> None:
    flattened = np.asarray(values, dtype=np.int32).flatten(order="C")

    with output_path.open("w", encoding="utf-8", newline="\n") as file:
        for value in flattened:
            file.write(int32_to_hex(int(value)) + "\n")


def verify_shape(name: str, values: np.ndarray) -> None:
    expected_shape = EXPECTED_SHAPES[name]

    if tuple(values.shape) != expected_shape:
        raise ValueError(
            f"{name} has shape {values.shape}, expected {expected_shape}"
        )


def fixed_point_multiplier(floating_multiplier: float, shift: int) -> int:
    if floating_multiplier < 0:
        raise ValueError("Requantization multiplier must be non-negative")

    integer_multiplier = int(round(floating_multiplier * (1 << shift)))

    if integer_multiplier > (2**31 - 1):
        raise OverflowError(
            f"Multiplier {integer_multiplier} does not fit in signed 32 bits"
        )

    return integer_multiplier


def export_model_memories(quantized_model) -> dict[str, dict[str, int | float]]:
    multiplier_metadata: dict[str, dict[str, int | float]] = {}

    for layer_name in ALL_COMPUTE_LAYERS:
        weight_name = f"{layer_name}_weight"
        bias_name = f"{layer_name}_bias"
        multiplier_name = f"{layer_name}_multiplier"

        if weight_name not in quantized_model:
            raise KeyError(f"Missing tensor: {weight_name}")
        if bias_name not in quantized_model:
            raise KeyError(f"Missing tensor: {bias_name}")
        if multiplier_name not in quantized_model:
            raise KeyError(f"Missing scalar: {multiplier_name}")

        weights = quantized_model[weight_name]
        biases = quantized_model[bias_name]

        verify_shape(weight_name, weights)
        verify_shape(bias_name, biases)

        weight_path = OUTPUT_DIRECTORY / f"{layer_name}_weights.mem"
        bias_path = OUTPUT_DIRECTORY / f"{layer_name}_biases.mem"

        write_int8_mem(weight_path, weights)
        write_int32_mem(bias_path, biases)

        floating_multiplier = float(quantized_model[multiplier_name])
        integer_multiplier = fixed_point_multiplier(floating_multiplier, REQUANT_SHIFT)

        multiplier_metadata[layer_name] = {
            "floating_multiplier": floating_multiplier,
            "integer_multiplier": integer_multiplier,
            "integer_multiplier_hex": uint32_to_hex(integer_multiplier),
            "shift": REQUANT_SHIFT,
            "weight_count": int(weights.size),
            "bias_count": int(biases.size),
        }

        print(
            f"{layer_name:5s} | weights={weights.size:6d} | "
            f"biases={biases.size:3d} | multiplier={floating_multiplier:.10f} | "
            f"Q{REQUANT_SHIFT}={integer_multiplier}"
        )

    return multiplier_metadata


def export_input_image() -> int:
    input_path = INTEGER_RESULTS_DIRECTORY / "input.npy"

    if not input_path.exists():
        raise FileNotFoundError(
            f"Missing integer input: {input_path}\nRun integer_reference.py first."
        )

    input_tensor = np.load(input_path)

    expected_shape = (1, 3, 32, 32)

    if tuple(input_tensor.shape) != expected_shape:
        raise ValueError(
            f"Input tensor has shape {input_tensor.shape}, expected {expected_shape}"
        )

    # Layout: [channel][row][column]
    input_chw = input_tensor[0]

    output_path = OUTPUT_DIRECTORY / "input_image_horse.mem"
    write_int8_mem(output_path, input_chw)

    return int(input_chw.size)


def export_expected_logits() -> list[int]:
    logits_path = INTEGER_RESULTS_DIRECTORY / "fc8.npy"

    if not logits_path.exists():
        raise FileNotFoundError(
            f"Missing expected logits: {logits_path}\nRun integer_reference.py first."
        )

    logits = np.load(logits_path)

    expected_shape = (1, NUM_CLASSES)

    if tuple(logits.shape) != expected_shape:
        raise ValueError(
            f"FC8 logits have shape {logits.shape}, expected {expected_shape}"
        )

    logits_vector = logits[0].astype(np.int8)

    write_int8_mem(OUTPUT_DIRECTORY / "expected_logits_horse.mem", logits_vector)

    predicted_class = int(np.argmax(logits_vector))

    with (OUTPUT_DIRECTORY / "expected_prediction_horse.txt").open(
        "w", encoding="utf-8", newline="\n"
    ) as file:
        file.write(f"{predicted_class}\n")

    return [int(value) for value in logits_vector]


def export_verilog_include(multiplier_metadata: dict[str, dict[str, int | float]]) -> None:
    output_path = OUTPUT_DIRECTORY / "cifar_requantization.vh"

    with output_path.open("w", encoding="utf-8", newline="\n") as file:
        file.write("`ifndef CIFAR_REQUANTIZATION_VH\n")
        file.write("`define CIFAR_REQUANTIZATION_VH\n\n")
        file.write(f"`define CIFAR_REQUANT_SHIFT {REQUANT_SHIFT}\n\n")

        for layer_name in ALL_COMPUTE_LAYERS:
            macro_name = layer_name.upper()
            integer_multiplier = int(multiplier_metadata[layer_name]["integer_multiplier"])
            file.write(f"`define {macro_name}_REQUANT_MULTIPLIER 32'd{integer_multiplier}\n")

        file.write("\n`endif\n")


def export_manifest(
    multiplier_metadata: dict[str, dict[str, int | float]],
    input_count: int,
    expected_logits: list[int],
    quantization_configuration: dict,
) -> None:
    manifest = {
        "network": "CIFAR-10 ten-class AlexNet-style CNN",
        "input": {
            "shape": [3, 32, 32],
            "layout": "channel-row-column",
            "count": input_count,
            "data_type": "signed_int8",
            "memory_file": "input_image_horse.mem",
        },
        "classes": CLASS_NAMES,
        "expected_logits": expected_logits,
        "expected_class": int(np.argmax(np.asarray(expected_logits))),
        "requantization": {
            "format": f"Q0.{REQUANT_SHIFT} integer multiplier",
            "shift": REQUANT_SHIFT,
            "layers": multiplier_metadata,
        },
        "quantization_configuration": quantization_configuration,
    }

    output_path = OUTPUT_DIRECTORY / "rtl_manifest.json"

    with output_path.open("w", encoding="utf-8") as file:
        json.dump(manifest, file, indent=2)


def count_file_lines(path: Path) -> int:
    with path.open("r", encoding="utf-8") as file:
        return sum(1 for _ in file)


def validate_exported_files(
    multiplier_metadata: dict[str, dict[str, int | float]],
    input_count: int,
) -> None:
    expected_line_counts = {
        "input_image_horse.mem": input_count,
        "expected_logits_horse.mem": NUM_CLASSES,
    }

    for layer_name in ALL_COMPUTE_LAYERS:
        expected_line_counts[f"{layer_name}_weights.mem"] = int(
            multiplier_metadata[layer_name]["weight_count"]
        )
        expected_line_counts[f"{layer_name}_biases.mem"] = int(
            multiplier_metadata[layer_name]["bias_count"]
        )

    for filename, expected_count in expected_line_counts.items():
        path = OUTPUT_DIRECTORY / filename
        actual_count = count_file_lines(path)

        if actual_count != expected_count:
            raise RuntimeError(
                f"{filename}: expected {expected_count} lines, found {actual_count}"
            )

    print()
    print("All exported file lengths verified.")


def main() -> None:
    if not QUANTIZED_MODEL_PATH.exists():
        raise FileNotFoundError(f"Missing quantized model: {QUANTIZED_MODEL_PATH}")

    if not QUANTIZATION_CONFIG_PATH.exists():
        raise FileNotFoundError(f"Missing quantization configuration: {QUANTIZATION_CONFIG_PATH}")

    OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)

    quantized_model = np.load(QUANTIZED_MODEL_PATH)

    with QUANTIZATION_CONFIG_PATH.open("r", encoding="utf-8") as file:
        quantization_configuration = json.load(file)

    print()
    print("========================================")
    print("Exporting RTL memory files")
    print("========================================")

    multiplier_metadata = export_model_memories(quantized_model)

    input_count = export_input_image()
    expected_logits = export_expected_logits()

    export_verilog_include(multiplier_metadata)

    export_manifest(
        multiplier_metadata=multiplier_metadata,
        input_count=input_count,
        expected_logits=expected_logits,
        quantization_configuration=quantization_configuration,
    )

    validate_exported_files(multiplier_metadata, input_count)

    predicted_class = int(np.argmax(expected_logits))

    print()
    print("========================================")
    print("RTL export complete")
    print(f"Output directory: {OUTPUT_DIRECTORY}")
    print(f"Input values: {input_count}")
    print(f"Expected logits: {expected_logits}")
    print(f"Expected class: {predicted_class} ({CLASS_NAMES[predicted_class]})")
    print("========================================")


if __name__ == "__main__":
    main()